In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from functools import lru_cache

from cl import RecordingView
import numpy as np
import datetime
from h5py import h5

In [ ]:

runs = sorted(list(Path('/mnt/data/cl/cl-pong/43492').glob('*')))
runs

In [ ]:

def count_events(gamestate):
    timesamps = sorted(gamestate.keys())
    events = [0,0]
    for t in timesamps:
        if gamestate[t]['game_state'] == 'BALL_HIT_PADDLE':
            events[0] += 1
        elif gamestate[t]['game_state'] == 'BALL_MISS':
            events[1] += 1
    return events

def get_event_timestamps(gamestate):
    events = {'BALL_HIT_PADDLE':[], 'BALL_MISS':[]}

    for t in sorted(gamestate.keys()):
        state = gamestate[t]['game_state']
        if state in events:
            events[state].append(t)

    return [events['BALL_HIT_PADDLE'], events['BALL_MISS']]


@lru_cache()
def f(runs):
    configs = []
    game1 = []
    rest = []
    game2 = []
    dates = []
    ts1 = []
    ts2 = []
    for i,run in enumerate(tqdm(runs)):
        sub_recordings = sorted(list(run.glob('*.h5')))
        if len(sub_recordings) != 3:
            continue

        date = datetime.datetime.strptime(run.name.split('+')[0], '%Y-%m-%d_%H-%M-%S.%f')
        dates.append(date)

        game1.append(RecordingView(str(sub_recordings[0])))
        rest.append(RecordingView(str(sub_recordings[1])))
        game2.append(RecordingView(str(sub_recordings[2])))

        ts1.append(get_event_timestamps(game1[-1].data_streams.gamestate))
        ts2.append(get_event_timestamps(game2[-1].data_streams.gamestate))

        with open(run / 'config.json') as fhan:
            configs.append(json.load(fhan))


    scores1 = np.array([[len(x) for x in t] for t in ts1])
    scores2 = np.array([[len(x) for x in t] for t in ts2])
    return configs, game1, rest, game2, dates, ts1, ts2, scores1, scores2

configs, game1, rest, game2, dates, ts1, ts2, scores1, scores2 = f(tuple(runs))

In [ ]:
r = RecordingView(str(list(runs[0].glob('*.h5'))[0]))

In [ ]:
plt.plot(dates, scores1, '.-')
plt.legend(['ball hits paddle count', 'ball misses paddle count'])
plt.gcf().autofmt_xdate()
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))


In [ ]:
plt.plot(dates,scores1[:,0] / scores1.sum(axis=1), '.-', label='ball hits/all trials', )
plt.legend()
plt.ylim([0,1])

plt.gcf().autofmt_xdate()
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))


In [ ]:

hits,misses = list(zip(*ts2))

hits = np.sort(np.hstack(hits)) / r.attributes['frames_per_second']
misses = np.sort(np.hstack(misses))/ r.attributes['frames_per_second']

bins = np.arange(min(hits.min(), misses.min()), max(hits.max(), misses.max()), 60)

hit_counts, _, _ = plt.hist(hits, bins=bins, alpha=.5, label='hits');
miss_counts, _, _ = plt.hist(misses, bins=bins, alpha=.5, label='misses');

plt.xlabel('seconds into the trial')
plt.legend()

In [ ]:
bins[-1]

In [ ]:
plt.plot(hit_counts / (miss_counts + hit_counts))